# 0. Data Loading

In [14]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

drive.mount("/content/drive")

# 自动定位数据集
root = Path("/content/drive/MyDrive/Amazon/baby")
matches = list(root.rglob("baby.inter"))
assert matches, "未找到 baby.inter，请检查 Google Drive 路径"
data_dir = matches[0].parent

# 读取数据
inter = pd.read_csv(data_dir / "baby.inter", sep="\t")
user_map = pd.read_csv(data_dir / "u_id_mapping.csv", sep="\t")
item_map = pd.read_csv(data_dir / "i_id_mapping.csv", sep="\t")
image_feat = np.load(data_dir / "image_feat.npy", mmap_mode="r")
text_feat = np.load(data_dir / "text_feat.npy", mmap_mode="r")
user_graph = np.load(data_dir / "user_graph_dict.npy", allow_pickle=True)
user_graph = user_graph.item() if user_graph.shape == () else user_graph

# 数据集概览
summary = pd.DataFrame([
    ["baby.inter", str(inter.shape), "用户-商品交互", "用户ID、商品ID、评分、时间戳、x_label", "训练与评估推荐模型"],
    ["u_id_mapping.csv", str(user_map.shape), "用户ID映射", "Amazon原始用户ID → 数字ID", "用户编号转换"],
    ["i_id_mapping.csv", str(item_map.shape), "商品ID映射", "Amazon ASIN → 数字ID", "商品编号转换"],
    ["image_feat.npy", str(image_feat.shape), "商品图片特征", "每件商品的4096维视觉向量", "多模态推荐、视觉相似度"],
    ["text_feat.npy", str(text_feat.shape), "商品文本特征", "每件商品的384维文本向量", "多模态推荐、语义相似度"],
    ["user_graph_dict.npy", f"{len(user_graph)} keys" if isinstance(user_graph, dict) else str(user_graph.shape),
     "用户关系结构", "以用户ID为键的嵌套关系数据（具体语义待确认）", "图结构分析、GNN研究"]
], columns=["File", "Shape", "Type", "Description", "Usage"])

print(f"Dataset: {data_dir}")
print(f"Users: {inter['userID'].nunique():,} | Items: {inter['itemID'].nunique():,} | Interactions: {len(inter):,}")
display(summary)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset: /content/drive/MyDrive/Amazon/baby
Users: 19,445 | Items: 7,050 | Interactions: 160,792


,File,Shape,Type,Description,Usage
0,baby.inter,"(160792, 5)",用户-商品交互,用户ID、商品ID、评分、时间戳、x_label,训练与评估推荐模型
1,u_id_mapping.csv,"(19445, 2)",用户ID映射,Amazon原始用户ID → 数字ID,用户编号转换
2,i_id_mapping.csv,"(7050, 2)",商品ID映射,Amazon ASIN → 数字ID,商品编号转换
3,image_feat.npy,"(7050, 4096)",商品图片特征,每件商品的4096维视觉向量,多模态推荐、视觉相似度
4,text_feat.npy,"(7050, 384)",商品文本特征,每件商品的384维文本向量,多模态推荐、语义相似度
5,user_graph_dict.npy,19445 keys,用户关系结构,以用户ID为键的嵌套关系数据（具体语义待确认）,图结构分析、GNN研究


# 1. Retrieval

## 1.1 ItemCF

In [15]:
## 离线

import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, diags

# 1. Dataset split
train = inter.loc[inter["x_label"] == 0].copy()
valid = inter.loc[inter["x_label"] == 1].copy()
test = inter.loc[inter["x_label"] == 2].copy()

n_users = int(inter["userID"].max()) + 1
n_items = int(inter["itemID"].max()) + 1

# 2. Experiment configurations
experiments = {
    f"itemcf-{mode}-nb{nb}": {"mode": mode, "nb": nb}
    for mode in ["binary", "rating"]
    for nb in [10, 50, 100, "all"]
}

# 3. Build interaction matrix
def build_matrix(df, mode):
    weights = (
        np.ones(len(df), dtype=np.float32)
        if mode == "binary"
        else df["rating"].to_numpy(dtype=np.float32)
    )
    return csr_matrix(
        (weights, (df["userID"], df["itemID"])),
        shape=(n_users, n_items),
        dtype=np.float32
    )

# 4. Compute cosine similarity and Top-N neighbors
def build_item_index(R, nb):
    norms = np.sqrt(np.asarray(R.power(2).sum(axis=0)).ravel())
    inv_norms = np.divide(
        1.0, norms,
        out=np.zeros_like(norms),
        where=norms > 0
    )

    X = R @ diags(inv_norms)
    S = (X.T @ X).tocsr()
    S.setdiag(0)
    S.eliminate_zeros()
    S.sort_indices()

    if nb == "all":
        return S

    rows, cols, values = [], [], []

    for i in range(n_items):
        start, end = S.indptr[i], S.indptr[i + 1]
        ids = S.indices[start:end]
        scores = S.data[start:end]

        if len(scores) > nb:
            keep = np.argpartition(-scores, nb - 1)[:nb]
            ids, scores = ids[keep], scores[keep]

        order = np.lexsort((ids, -scores))
        rows.extend([i] * len(order))
        cols.extend(ids[order])
        values.extend(scores[order])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=(n_items, n_items)
    )

# 5. Offline training
offline_models = {}
offline_records = []

for name, cfg in experiments.items():
    start = time.perf_counter()

    R = build_matrix(train, cfg["mode"])
    S = build_item_index(R, cfg["nb"])

    elapsed = time.perf_counter() - start

    offline_models[name] = {"R": R, "S": S}
    offline_records.append({
        "Experiment": name,
        "OfflineTime(s)": round(elapsed, 3),
        "IndexEdges": S.nnz,
        "IndexMemory(MB)": round(
            (S.data.nbytes + S.indices.nbytes + S.indptr.nbytes) / 1024**2, 3
        )
    })

offline_df = pd.DataFrame(offline_records)

print(f"Users: {n_users:,} | Items: {n_items:,}")
print(f"Train: {len(train):,} | Validation: {len(valid):,} | Test: {len(test):,}")
print("\nOffline computation completed:")
display(offline_df)

Users: 19,445 | Items: 7,050
Train: 118,551 | Validation: 20,559 | Test: 21,682

Offline computation completed:


,Experiment,OfflineTime(s),IndexEdges,IndexMemory(MB)
0,itemcf-binary-nb10,0.370,70117,0.562
1,itemcf-binary-nb50,0.558,300620,2.320
2,itemcf-binary-nb100,0.704,467026,3.590
3,itemcf-binary-nball,0.087,903922,6.923
4,itemcf-rating-nb10,0.366,70117,0.562
5,itemcf-rating-nb50,0.557,300620,2.320
6,itemcf-rating-nb100,0.483,467026,3.590
7,itemcf-rating-nball,0.065,903922,6.923


In [16]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

RECALL_TOPK = 100
RECALL_DIR = data_dir / "recall"
RECALL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Online retrieval
def retrieve_itemcf(R, S, topk=100):
    recommendations = {}
    scored_recommendations = {}
    candidate_counts = {}
    start = time.perf_counter()

    for u in range(R.shape[0]):
        history = R.getrow(u)
        scores = np.asarray((history @ S).toarray()).ravel()
        scores[history.indices] = -np.inf

        candidates = np.flatnonzero((scores > 0) & np.isfinite(scores))
        candidate_counts[u] = len(candidates)

        if len(candidates) > topk:
            chosen = np.argpartition(-scores[candidates], topk - 1)[:topk]
            candidates = candidates[chosen]

        order = np.lexsort((candidates, -scores[candidates]))
        ranked = candidates[order]

        recommendations[u] = ranked.tolist()
        scored_recommendations[u] = [
            (int(i), float(scores[i])) for i in ranked
        ]

    elapsed = time.perf_counter() - start
    return recommendations, scored_recommendations, candidate_counts, elapsed

# 2. Run experiments and save recall results
online_results = {}
scored_results = {}
online_records = []

for name, model in offline_models.items():
    recs, scored_recs, counts, elapsed = retrieve_itemcf(
        model["R"], model["S"], topk=RECALL_TOPK
    )

    online_results[name] = recs
    scored_results[name] = scored_recs

    rows = [
        (u, item, score, rank, name)
        for u, items in scored_recs.items()
        for rank, (item, score) in enumerate(items, start=1)
    ]

    recall_df = pd.DataFrame(
        rows,
        columns=["userID", "itemID", "score", "rank", "channel"]
    )

    recall_df.to_parquet(
        RECALL_DIR / f"{name}.parquet",
        index=False
    )

    online_records.append({
        "Experiment": name,
        "OnlineTime(s)": round(elapsed, 3),
        "Latency(ms/user)": round(elapsed / n_users * 1000, 4),
        "Users/s": round(n_users / elapsed, 2),
        "AvgCandidates": round(np.mean(list(counts.values())), 2),
        "UsersWith100": sum(c >= RECALL_TOPK for c in counts.values()),
        "EmptyUsers": sum(c == 0 for c in counts.values())
    })

online_df = pd.DataFrame(online_records)

print(f"ItemCF online retrieval completed | Top-{RECALL_TOPK}")
print(f"Saved to: {RECALL_DIR}")
display(online_df)

# 3. Example recommendations
example_user = 0
example_name = "itemcf-rating-nb50"

print(f"\nExperiment: {example_name}")
print(f"User: {example_user}")
print(f"History: {offline_models[example_name]['R'].getrow(example_user).indices.tolist()}")
print(f"Top-20: {scored_results[example_name][example_user][:20]}")

ItemCF online retrieval completed | Top-100
Saved to: /content/drive/MyDrive/Amazon/baby/recall


,Experiment,OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
0,itemcf-binary-nb10,4.015,0.2065,4843.64,55.29,1913,0
1,itemcf-binary-nb50,6.360,0.3271,3057.35,261.13,19240,0
2,itemcf-binary-nb100,5.244,0.2697,3708.31,472.18,19410,0
3,itemcf-binary-nball,8.475,0.4358,2294.49,1592.00,19410,0
4,itemcf-rating-nb10,4.027,0.2071,4828.71,55.50,1951,0
5,itemcf-rating-nb50,6.491,0.3338,2995.81,261.84,19243,0
6,itemcf-rating-nb100,6.651,0.3420,2923.57,471.77,19410,0
7,itemcf-rating-nball,6.270,0.3224,3101.43,1592.00,19410,0



Experiment: itemcf-rating-nb50
User: 0
History: [0, 1587, 1879]
Top-20: [(880, 1.102102518081665), (4511, 1.0206207036972046), (5403, 0.9660072326660156), (2491, 0.9021097421646118), (638, 0.8783508539199829), (4144, 0.8020734786987305), (6165, 0.7524115443229675), (6244, 0.7524115443229675), (1768, 0.7245919704437256), (1663, 0.7107168436050415), (386, 0.6789067983627319), (3131, 0.6255632638931274), (5125, 0.6143414974212646), (6379, 0.6143414378166199), (5166, 0.610584020614624), (6253, 0.5977281332015991), (6145, 0.5572781562805176), (4354, 0.5252286195755005), (5058, 0.5183911919593811), (1728, 0.5148611664772034)]


In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

EVAL_K = [5, 10, 20, 50, 100]

# 1. Build ground truth
def build_ground_truth(df):
    return {
        int(u): set(items)
        for u, items in df.groupby("userID")["itemID"]
    }

valid_truth = build_ground_truth(valid)
test_truth = build_ground_truth(test)

# 2. Training popularity for novelty
item_pop = np.bincount(
    train["itemID"].to_numpy(dtype=int),
    minlength=n_items
)
item_prob = (item_pop + 1) / (n_users + 1)
item_novelty = -np.log2(item_prob)

# 3. Ranking metrics
def evaluate(recommendations, ground_truth, ks=EVAL_K):
    records = []
    users = sorted(ground_truth)
    catalog = set(np.flatnonzero(item_pop > 0))
    catalog_size = len(catalog)

    for k in ks:
        recalls, ndcgs, precisions = [], [], []
        hitrates, maps, mrrs, novelties = [], [], [], []
        covered_items = set()

        for u in users:
            truth = ground_truth[u]
            recs = recommendations.get(u, [])[:k]

            if not truth:
                continue

            hits = np.array([int(i in truth) for i in recs])
            n_hits = int(hits.sum())
            positions = np.arange(1, len(recs) + 1)

            # Recall and Precision
            recalls.append(n_hits / len(truth))
            precisions.append(n_hits / k)
            hitrates.append(float(n_hits > 0))

            # NDCG
            dcg = np.sum(hits / np.log2(positions + 1))
            ideal_len = min(len(truth), k)
            idcg = np.sum(
                1 / np.log2(np.arange(2, ideal_len + 2))
            )
            ndcgs.append(dcg / idcg)

            # MAP
            cumulative_hits = np.cumsum(hits)
            ap = np.sum(
                (cumulative_hits / positions) * hits
            ) / min(len(truth), k)
            maps.append(ap)

            # MRR
            hit_positions = np.flatnonzero(hits)
            mrrs.append(
                1 / (hit_positions[0] + 1)
                if len(hit_positions) else 0.0
            )

            # Coverage and Novelty
            covered_items.update(recs)
            if recs:
                novelties.append(np.mean(item_novelty[recs]))

        records.append({
            "K": k,
            "Recall": np.mean(recalls),
            "NDCG": np.mean(ndcgs),
            "Precision": np.mean(precisions),
            "HitRate": np.mean(hitrates),
            "MAP": np.mean(maps),
            "MRR": np.mean(mrrs),
            "Coverage": len(covered_items & catalog) / catalog_size,
            "Novelty": np.mean(novelties) if novelties else np.nan
        })

    return pd.DataFrame(records)

# 4. Evaluate all experiments
results = []

for name, recs in online_results.items():
    for split_name, truth in [
        ("Validation", valid_truth),
        ("Test", test_truth)
    ]:
        metrics = evaluate(recs, truth)
        metrics.insert(0, "Split", split_name)
        metrics.insert(0, "Experiment", name)
        results.append(metrics)

results_df = pd.concat(results, ignore_index=True)

# 5. Combine accuracy and efficiency
results_df = (
    results_df
    .merge(offline_df, on="Experiment", how="left")
    .merge(online_df, on="Experiment", how="left")
)

# 6. Save results
output_dir = Path("/content/drive/MyDrive/RecSys-Lab/results")
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "itemcf_all_metrics.csv",
    index=False
)

# 7. Show validation comparison
core_metrics = [
    "Experiment", "Recall", "NDCG", "Precision",
    "HitRate", "MAP", "MRR", "Coverage", "Novelty"
]

validation_20 = (
    results_df[
        (results_df["Split"] == "Validation") &
        (results_df["K"] == 20)
    ]
    .sort_values("Recall", ascending=False)
)

print("ItemCF | Validation | Top-20")
display(validation_20[core_metrics].round(4))

print("\nBest configuration by Validation Recall@20:")
best_name = validation_20.iloc[0]["Experiment"]
print(best_name)

print("\nItemCF | Test | Top-20 (Validation-selected model)")
best_test = results_df[
    (results_df["Split"] == "Test") &
    (results_df["K"] == 20) &
    (results_df["Experiment"] == best_name)
]
display(best_test[core_metrics].round(4))

print(f"\nSaved: {output_dir / 'itemcf_all_metrics.csv'}")

ItemCF | Validation | Top-20


,Experiment,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty
32,itemcf-binary-nball,0.0409,0.0189,0.0023,0.0446,0.0124,0.0134,0.9999,10.8678
72,itemcf-rating-nball,0.0382,0.0172,0.0021,0.0415,0.0111,0.0120,1.0000,10.9013
22,itemcf-binary-nb100,0.0317,0.0150,0.0017,0.0335,0.0101,0.0107,0.9928,11.2422
62,itemcf-rating-nb100,0.0314,0.0143,0.0017,0.0333,0.0094,0.0100,0.9974,11.1798
12,itemcf-binary-nb50,0.0307,0.0143,0.0017,0.0326,0.0095,0.0102,0.9854,11.3198
52,itemcf-rating-nb50,0.0291,0.0134,0.0016,0.0309,0.0088,0.0093,0.9936,11.2469
2,itemcf-binary-nb10,0.0288,0.0135,0.0015,0.0306,0.0091,0.0096,0.9649,11.4236
42,itemcf-rating-nb10,0.0272,0.0122,0.0015,0.0289,0.0079,0.0083,0.9837,11.3457



Best configuration by Validation Recall@20:
itemcf-binary-nball

ItemCF | Test | Top-20 (Validation-selected model)


,Experiment,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty
37,itemcf-binary-nball,0.0408,0.0191,0.0024,0.0474,0.0123,0.0141,0.9999,10.8678



Saved: /content/drive/MyDrive/RecSys-Lab/results/itemcf_all_metrics.csv


## 1.2 Swing

In [18]:
from collections import defaultdict
from itertools import combinations
from scipy.sparse import csr_matrix
import numpy as np
import pandas as pd
import time

SWING_ALPHA = 1.0
SWING_NEIGHBORS = [10, 50, 100]
SWING_MODES = ["binary", "rating"]

# 1. Build binary interaction matrix
R_binary = build_matrix(train, "binary").tocsr()
R_binary.sort_indices()

# 2. Compute Swing similarity
def build_swing_similarity(R, alpha=1.0):
    item_scores = defaultdict(float)
    user_items = [
        R.indices[R.indptr[u]:R.indptr[u + 1]]
        for u in range(R.shape[0])
    ]

    item_users = R.T.tocsr()
    user_pairs = set()

    for i in range(R.shape[1]):
        users = item_users.indices[
            item_users.indptr[i]:item_users.indptr[i + 1]
        ]
        user_pairs.update(combinations(users, 2))

    for u, v in user_pairs:
        common = np.intersect1d(
            user_items[u], user_items[v],
            assume_unique=True
        )

        if len(common) < 2:
            continue

        weight = 1.0 / (alpha + len(common))

        for i, j in combinations(common, 2):
            item_scores[(int(i), int(j))] += weight

    rows, cols, values = [], [], []

    for (i, j), score in item_scores.items():
        rows.extend([i, j])
        cols.extend([j, i])
        values.extend([score, score])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=(R.shape[1], R.shape[1])
    )

# 3. Keep Top-N item neighbors
def truncate_neighbors(S, nb):
    rows, cols, values = [], [], []

    for i in range(S.shape[0]):
        start, end = S.indptr[i], S.indptr[i + 1]
        ids = S.indices[start:end]
        scores = S.data[start:end]

        if len(scores) > nb:
            keep = np.argpartition(-scores, nb - 1)[:nb]
            ids, scores = ids[keep], scores[keep]

        order = np.lexsort((ids, -scores))
        rows.extend([i] * len(order))
        cols.extend(ids[order])
        values.extend(scores[order])

    return csr_matrix(
        (np.asarray(values, dtype=np.float32), (rows, cols)),
        shape=S.shape
    )

# 4. Build offline indexes
offline_models = {}
offline_records = []

start = time.perf_counter()
S_full = build_swing_similarity(R_binary, SWING_ALPHA)
similarity_time = time.perf_counter() - start

for nb in SWING_NEIGHBORS:
    start = time.perf_counter()
    S = truncate_neighbors(S_full, nb)
    index_time = time.perf_counter() - start

    for mode in SWING_MODES:
        name = f"swing-{mode}-a{SWING_ALPHA:g}-nb{nb}"
        R = build_matrix(train, mode)

        offline_models[name] = {"R": R, "S": S}
        offline_records.append({
            "Experiment": name,
            "OfflineTime(s)": round(similarity_time + index_time, 3),
            "IndexEdges": S.nnz,
            "IndexMemory(MB)": round(
                (S.data.nbytes + S.indices.nbytes + S.indptr.nbytes) / 1024**2, 3
            )
        })

offline_df = pd.DataFrame(offline_records)

print(f"Swing similarity computation: {similarity_time:.2f}s")
print(f"Non-zero similarities: {S_full.nnz:,}")
display(offline_df)

Swing similarity computation: 31.36s
Non-zero similarities: 90,578


,Experiment,OfflineTime(s),IndexEdges,IndexMemory(MB)
0,swing-binary-a1-nb10,31.439,24430,0.213
1,swing-rating-a1-nb10,31.439,24430,0.213
2,swing-binary-a1-nb50,31.447,55553,0.451
3,swing-rating-a1-nb50,31.447,55553,0.451
4,swing-binary-a1-nb100,31.452,70608,0.566
5,swing-rating-a1-nb100,31.452,70608,0.566


In [19]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

RECALL_TOPK = 100
RECALL_DIR = data_dir / "recall"
RECALL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Online retrieval and save candidates
online_results = {}
scored_results = {}
online_records = []

for name, model in offline_models.items():
    recs, scored_recs, counts, elapsed = retrieve_itemcf(
        model["R"], model["S"], topk=RECALL_TOPK
    )

    online_results[name] = recs
    scored_results[name] = scored_recs

    rows = [
        (u, item, score, rank, name)
        for u, items in scored_recs.items()
        for rank, (item, score) in enumerate(items, start=1)
    ]

    recall_df = pd.DataFrame(
        rows,
        columns=["userID", "itemID", "score", "rank", "channel"]
    )

    recall_df.to_parquet(
        RECALL_DIR / f"{name}.parquet",
        index=False
    )

    online_records.append({
        "Experiment": name,
        "OnlineTime(s)": round(elapsed, 3),
        "Latency(ms/user)": round(elapsed / n_users * 1000, 4),
        "Users/s": round(n_users / elapsed, 2),
        "AvgCandidates": round(np.mean(list(counts.values())), 2),
        "UsersWith100": sum(c >= RECALL_TOPK for c in counts.values()),
        "EmptyUsers": sum(c == 0 for c in counts.values())
    })

online_df = pd.DataFrame(online_records)

print(f"Swing online retrieval completed | Top-{RECALL_TOPK}")
print(f"Saved to: {RECALL_DIR}")
display(online_df)

Swing online retrieval completed | Top-100
Saved to: /content/drive/MyDrive/Amazon/baby/recall


,Experiment,OnlineTime(s),Latency(ms/user),Users/s,AvgCandidates,UsersWith100,EmptyUsers
0,swing-binary-a1-nb10,3.998,0.2056,4864.14,33.39,288,32
1,swing-rating-a1-nb10,5.148,0.2648,3776.97,33.39,288,32
2,swing-binary-a1-nb50,4.733,0.2434,4108.04,110.70,9426,32
3,swing-rating-a1-nb50,6.400,0.3291,3038.38,110.70,9426,32
4,swing-binary-a1-nb100,5.227,0.2688,3720.40,169.81,14701,32
5,swing-rating-a1-nb100,6.832,0.3513,2846.24,169.81,14701,32


In [20]:
from pathlib import Path

results = []

for name, recs in online_results.items():
    for split_name, truth in [
        ("Validation", valid_truth),
        ("Test", test_truth)
    ]:
        metrics = evaluate(recs, truth)
        metrics.insert(0, "Split", split_name)
        metrics.insert(0, "Experiment", name)
        results.append(metrics)

swing_results = pd.concat(results, ignore_index=True)

swing_results = (
    swing_results
    .merge(offline_df, on="Experiment", how="left")
    .merge(online_df, on="Experiment", how="left")
)

output_dir = Path("/content/drive/MyDrive/RecSys-Lab/results")
output_dir.mkdir(parents=True, exist_ok=True)

swing_results.to_csv(
    output_dir / "swing_all_metrics.csv",
    index=False
)

validation_20 = (
    swing_results[
        (swing_results["Split"] == "Validation") &
        (swing_results["K"] == 20)
    ]
    .sort_values("Recall", ascending=False)
)

print("Swing | Validation | Top-20")
display(validation_20[[
    "Experiment", "Recall", "NDCG", "Precision",
    "HitRate", "MAP", "MRR", "Coverage", "Novelty"
]].round(4))

best_name = validation_20.iloc[0]["Experiment"]
print(f"\nBest Swing configuration: {best_name}")

print("\nTest | Top-20")
display(
    swing_results[
        (swing_results["Split"] == "Test") &
        (swing_results["K"] == 20) &
        (swing_results["Experiment"] == best_name)
    ][[
        "Experiment", "Recall", "NDCG", "Precision",
        "HitRate", "MAP", "MRR", "Coverage", "Novelty"
    ]].round(4)
)

print(f"\nSaved: {output_dir / 'swing_all_metrics.csv'}")

Swing | Validation | Top-20


,Experiment,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty
2,swing-binary-a1-nb10,0.0807,0.0373,0.0043,0.0851,0.0248,0.0259,0.3758,7.1900
12,swing-rating-a1-nb10,0.0806,0.0372,0.0043,0.0851,0.0247,0.0257,0.3775,7.1964
42,swing-binary-a1-nb100,0.0783,0.0368,0.0042,0.0823,0.0248,0.0258,0.3936,7.0458
22,swing-binary-a1-nb50,0.0779,0.0367,0.0041,0.0820,0.0247,0.0258,0.3934,7.0513
52,swing-rating-a1-nb100,0.0777,0.0365,0.0041,0.0819,0.0245,0.0256,0.4058,7.0554
32,swing-rating-a1-nb50,0.0775,0.0364,0.0041,0.0816,0.0245,0.0256,0.4057,7.0611



Best Swing configuration: swing-binary-a1-nb10

Test | Top-20


,Experiment,Recall,NDCG,Precision,HitRate,MAP,MRR,Coverage,Novelty
7,swing-binary-a1-nb10,0.0769,0.0361,0.0043,0.0851,0.0237,0.0258,0.3758,7.19



Saved: /content/drive/MyDrive/RecSys-Lab/results/swing_all_metrics.csv


## 1.3 UserCF